# report_data

> Build the data layer for the single-page evaluation report (#79): one aggregates-only
> JSON assembled from a pipeline output directory, plus the renderer that injects it into
> the self-contained HTML template. Replaces the render-time Quarto/papermill/SHAP path.


In [ ]:
#| default_exp report_data

In [ ]:
#| export
from __future__ import annotations

import json
import re
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import structlog

log = structlog.get_logger()

# Subgroup metrics are emitted only when the group has at least this many samples —
# below it the AUC is noise, and tiny groups edge toward re-identifiability.
MIN_SUBGROUP = 30
# Curves are thinned to this many points before embedding (endpoints always kept).
CURVE_POINTS = 120
# Number of equal-width probability bins for the calibration (reliability) diagram.
CALIBRATION_BINS = 10
# Decision-curve threshold grid (probability thresholds for net-benefit).
DCA_THRESHOLDS = np.linspace(0.02, 0.6, 30)

_CPU_MODELS = ("lr", "rf", "xgb")
_GPU_MODELS = ("tabpfn", "tabpfn_ft", "tabicl", "tabicl_ft")

In [ ]:
#| export
def _r(v, digits: int = 4):
    """Round a numeric value for embedding; None passes through."""
    return None if v is None else round(float(v), digits)


def _downsample(x, y, k: int = CURVE_POINTS):
    """Thin a curve to ~``k`` points, always keeping both endpoints."""
    n = len(x)
    if n <= k:
        return [_r(a) for a in x], [_r(b) for b in y]
    idx = np.unique(np.linspace(0, n - 1, k).astype(int))
    return [_r(x[i]) for i in idx], [_r(y[i]) for i in idx]


def assert_no_phi(blob: str) -> None:
    """Fail loud if the serialized report data carries anything sample-identifying.

    The upstream ``*_model_results.json`` files embed thousands of real MSK DMP sample
    ids (``oof_sample_ids``); the report must only ever ship aggregates. This is the
    hard guarantee, enforced at build time — not a convention.

    Raises:
        ValueError: if a DMP-shaped id or a sample-id field survived into the output.
    """
    hits = re.findall(r"P-\d{7}", blob)
    if hits:
        raise ValueError(
            f"PHI leak: {len(hits)} sample-id-shaped strings in report data — refusing to write"
        )
    if "sample_id" in blob or "SAMPLE_ID" in blob:
        raise ValueError("PHI leak: a sample-id field name survived into report data")

In [ ]:
#| export
def _build_cohort(labels: pd.DataFrame) -> dict:
    """Aggregate the labels table into the cohort tab (counts only, no identifiers)."""
    cohort = {
        "n_samples": int(len(labels)),
        "n_patients": int(labels["PATIENT_ID"].nunique()),
        "label_counts": {k: int(v) for k, v in labels["label"].value_counts().items()},
        "cancer_types": {
            k: int(v) for k, v in labels["CANCER_TYPE"].value_counts().head(14).items()
        },
        "cancer_types_other": int(
            labels["CANCER_TYPE"].value_counts().iloc[14:].sum()
            if labels["CANCER_TYPE"].nunique() > 14
            else 0
        ),
        "split_composition": {},
        "patients_in_both_splits": None,
    }
    if "split" in labels.columns:
        cohort["split_composition"] = {
            str(sp): {str(k): int(v) for k, v in grp["label"].value_counts().items()}
            for sp, grp in labels.groupby("split")
        }
        # ML-integrity check surfaced in the report: a sample-level split scatters a
        # patient's timepoints across train AND test, optimistically biasing holdout
        # metrics. Counted here, warned about client-side. See the split-leakage issue.
        cohort["patients_in_both_splits"] = int(
            (labels.groupby("PATIENT_ID")["split"].nunique() > 1).sum()
        )
        if cohort["patients_in_both_splits"]:
            log.warning(
                "split_patient_leakage",
                n_patients=cohort["patients_in_both_splits"],
                impact="holdout metrics optimistically biased",
            )
    return cohort

In [ ]:
#| export
def _oof_extras(mr: dict, best: str, lab_by_id: pd.DataFrame) -> dict:
    """Curves, calibration, decision curve and subgroup AUCs from the OOF arrays.

    Sample ids are joined to labels IN MEMORY and discarded — only per-group
    aggregates (n, n_pos, auc) are returned.
    """
    from sklearn.metrics import precision_recall_curve, roc_auc_score, roc_curve

    out: dict = {}
    probs = mr.get(f"{best}_oof_probs")
    y = mr.get("oof_labels")
    if not probs or not y or len(probs) != len(y):
        return out
    y_arr = np.asarray(y, dtype=int)
    p_arr = np.asarray(probs, dtype=float)

    fpr, tpr, _ = roc_curve(y_arr, p_arr)
    out["roc"] = dict(zip(("fpr", "tpr"), _downsample(fpr, tpr)))
    prec, recl, _ = precision_recall_curve(y_arr, p_arr)
    out["pr"] = dict(zip(("recall", "precision"), _downsample(recl[::-1], prec[::-1])))
    out["prevalence"] = _r(y_arr.mean())

    # sens@100spec_healthy is thresholded on the max score among the OOF healthy
    # normals — surface how many there are so the report can flag the variance.
    slabels = mr.get("oof_sample_labels") or []
    out["n_healthy_oof"] = int(sum(1 for s in slabels if s == "Healthy Normal"))

    bins = np.clip((p_arr * CALIBRATION_BINS).astype(int), 0, CALIBRATION_BINS - 1)
    cal = []
    for b in range(CALIBRATION_BINS):
        m = bins == b
        if m.sum() >= 10:
            cal.append(
                {"p_mean": _r(p_arr[m].mean()), "obs": _r(y_arr[m].mean()), "n": int(m.sum())}
            )
    out["calibration"] = cal

    n_all = len(y_arr)
    nb, nb_all = [], []
    for thr in DCA_THRESHOLDS:
        w = thr / (1 - thr)
        pred = p_arr >= thr
        tp = int((pred & (y_arr == 1)).sum())
        fp = int((pred & (y_arr == 0)).sum())
        nb.append(_r(tp / n_all - fp / n_all * w))
        nb_all.append(_r(y_arr.mean() - (1 - y_arr.mean()) * w))
    out["dca"] = {
        "thresholds": [_r(t) for t in DCA_THRESHOLDS],
        "net_benefit": nb,
        "treat_all": nb_all,
    }

    ids = mr.get("oof_sample_ids")
    if ids and len(ids) == len(y):
        sub: dict = defaultdict(dict)
        meta = lab_by_id.reindex(ids)
        for col, key in (("CANCER_TYPE", "cancer_type"), ("split", "split")):
            if col not in meta.columns:
                continue
            for grp, gidx in meta.groupby(col).groups.items():
                mask = meta.index.isin(gidx)
                yy, pp = y_arr[mask], p_arr[mask]
                if len(yy) < MIN_SUBGROUP or len(set(yy)) < 2:
                    continue
                sub[key][str(grp)] = {
                    "n": int(len(yy)),
                    "n_pos": int(yy.sum()),
                    "auc": _r(roc_auc_score(yy, pp)),
                }
        out["breakdowns"] = {
            k: dict(sorted(v.items(), key=lambda kv: -kv[1]["n"])[:12])
            for k, v in sub.items()
        }
    return out

In [ ]:
#| export
def _build_evaluator(row: pd.Series, outdir: Path, lab_by_id: pd.DataFrame) -> dict:
    """One evaluator record: scoreboard row + model detail + curves + ablation stability."""
    name = row["evaluator"]
    rec = {k: (_r(v) if isinstance(v, float) else v) for k, v in row.items()}

    mr: dict = {}
    for sub, fn in (("cpu", f"{name}_model_results.json"), ("gpu", f"{name}_gpu_model_results.json")):
        p = outdir / "models" / sub / fn
        if p.exists():
            data = json.loads(p.read_text())
            mr.update({k: v for k, v in data.items() if k != "oof_sample_ids"} if sub == "gpu" else data)
    if not mr:
        # Degrade AND surface: the scoreboard row still renders (with its status
        # column); the deep-dive sections are simply absent.
        log.warning("report_no_model_results", evaluator=name)
        return rec

    metrics = {}
    for m in [*_CPU_MODELS, *_GPU_MODELS]:
        if f"auc_{m}" not in mr:
            continue
        metrics[m] = {
            "auc": _r(mr.get(f"auc_{m}")),
            "ci": [_r(mr.get(f"auc_{m}_ci_lower")), _r(mr.get(f"auc_{m}_ci_upper"))],
            "auc_std": _r(mr.get(f"{m}_auc_std")),
            "fold_aucs": [_r(x) for x in mr.get(f"{m}_fold_aucs", [])],
            "sens95": _r(mr.get(f"{m}_sensitivity_at_95spec")),
            "sens99": _r(mr.get(f"{m}_sensitivity_at_99spec")),
            "sens100": _r(mr.get(f"{m}_sensitivity_at_100spec")),
            "sens100h": _r(mr.get(f"{m}_sensitivity_at_100spec_healthy")),
            "n_det100": mr.get(f"{m}_n_detected_at_100spec"),
            "n_pos": mr.get(f"{m}_n_total_positive"),
            "confusion": mr.get(f"{m}_confusion_matrix"),
            "holdout_auc": _r(mr.get(f"holdout_{m}_auc")),
            "holdout_sens100": _r(mr.get(f"holdout_{m}_sensitivity_at_100spec")),
        }
    rec["model_metrics"] = metrics

    best = row.get("best_model")
    rec["top_features"] = (mr.get(f"{best}_refit_features") or [])[:15]

    qc_path = outdir / "matrices" / "selected" / f"{name}_selection_qc.json"
    if qc_path.exists():
        qc = json.loads(qc_path.read_text())
        rec["selection"] = {
            "method": qc.get("method"),
            "n_input": qc.get("total_input_features"),
            "n_selected": qc.get("n_mrmr_selected") or qc.get("n_selected_union"),
            "n_variance_dropped": qc.get("n_variance_dropped"),
        }

    rec.update(_oof_extras(mr, best, lab_by_id))

    # Nested-CV feature-group ablation (ABLATE stage): winner group per outer fold,
    # per model — the selection-stability story. The per-sample fold_assignment array
    # in that file is deliberately never read into the output.
    bs_path = outdir / "ablation" / "merged" / f"{name}_best_subset.json"
    if bs_path.exists():
        bs = json.loads(bs_path.read_text())
        if not bs.get("passthrough"):
            fa = {}
            for m, md_ in (bs.get("per_model_per_fold_features") or {}).items():
                folds = md_.get("folds") or {}
                winners = Counter(
                    f.get("winner") for f in folds.values() if isinstance(f, dict)
                )
                scores = [
                    f.get("winner_score")
                    for f in folds.values()
                    if isinstance(f, dict) and f.get("winner_score") is not None
                ]
                fa[m] = {
                    "winners": dict(winners.most_common()),
                    "n_folds": len(folds),
                    "mean_winner_score": _r(np.mean(scores)) if scores else None,
                }
            rec["fold_ablation"] = {"groups": bs.get("groups"), "models": fa}
    return rec

In [ ]:
#| export
def _build_multimodal(mm_dir: Path) -> dict:
    """Multimodal tab: stacking/raw metrics per model + prep metadata (+ LOO ablation)."""
    prep_path = mm_dir / "prep_metadata.json"
    if not prep_path.exists():
        return {}
    prep = json.loads(prep_path.read_text())

    models: dict = {}
    for f in sorted(mm_dir.glob("stacking_*_results.json")):
        d = json.loads(f.read_text())
        mname = f.stem.replace("stacking_", "").replace("_results", "")
        entry = {}
        for scope in ("stacking", "raw"):
            auc = d.get(f"auc_{scope}_{mname}")
            if auc is None:
                continue
            entry[scope] = {
                "auc": _r(auc),
                "ci": [
                    _r(d.get(f"auc_{scope}_{mname}_ci_lower")),
                    _r(d.get(f"auc_{scope}_{mname}_ci_upper")),
                ],
                "fold_aucs": [_r(x) for x in d.get(f"{scope}_{mname}_fold_aucs", [])],
                "sens95": _r(d.get(f"{scope}_{mname}_sensitivity_at_95spec")),
                "sens100h": _r(d.get(f"{scope}_{mname}_sensitivity_at_100spec_healthy")),
                "vs_best_single": _r(d.get(f"{scope}_{mname}_vs_best_single")),
                "confusion": d.get(f"{scope}_{mname}_confusion_matrix"),
            }
            pr_c = d.get(f"{scope}_{mname}_pr_curve") or {}
            if isinstance(pr_c, dict) and "precision" in pr_c and "recall" in pr_c:
                rr, pp = _downsample(pr_c["recall"][::-1], pr_c["precision"][::-1])
                entry[scope]["pr"] = {"recall": rr, "precision": pp}
            fi = d.get(f"{scope}_{mname}_feature_importances")
            if isinstance(fi, dict):
                entry[scope]["top_importances"] = dict(
                    sorted(fi.items(), key=lambda kv: -abs(kv[1]))[:15]
                )
        models[mname] = entry

    ablation = None
    abl_path = mm_dir / "ablation_results.json"
    if abl_path.exists():
        abl = json.loads(abl_path.read_text())
        ablation = {
            "model": abl.get("ablation_model"),
            "baseline_auc": _r(abl.get("ablation_baseline_auc")),
            "fallback": abl.get("ablation_model_fallback"),
            "deltas": {
                k: {"delta": _r(v.get("delta")), "auc_without": _r(v.get("auc_without"))}
                for k, v in (abl.get("ablation") or {}).items()
                if isinstance(v, dict) and "delta" in v
            },
        }

    return {
        "models": models,
        "best_single_evaluator": prep.get("best_single_evaluator"),
        "best_single_auc": _r(prep.get("best_single_auc")),
        "selection": prep.get("multimodal_selection"),
        "n_evaluators": prep.get("n_evaluators"),
        "stacking_shape": prep.get("stacking_shape"),
        "single_evaluator_aucs": {
            k: _r(v) for k, v in (prep.get("single_evaluator_aucs") or {}).items()
        },
        "ablation": ablation,
    }

In [ ]:
#| export
def _build_diagnostics(trace_path: Path | None) -> dict:
    """Run diagnostics from the Nextflow execution trace, when one is available.

    The trace is written when the WORKFLOW finishes, so an in-pipeline report render
    cannot see it — the section is optional by design and fills in when ``kreview
    report`` is re-run on a completed output directory.
    """
    diag = {"processes": {}, "longest": [], "workflow": {}}
    if trace_path is None or not Path(trace_path).exists():
        return diag
    tr = pd.read_csv(trace_path, sep="\t")
    tr["proc"] = tr["process"].str.split(":").str[-1]
    for proc, grp in tr.groupby("proc"):
        diag["processes"][proc] = {
            "n": int(len(grp)),
            "completed": int((grp["status"] == "COMPLETED").sum()),
            "failed": int((grp["status"] == "FAILED").sum()),
            "max_attempt": int(grp["attempt"].max()),
        }
    done = tr[tr["status"] == "COMPLETED"]
    diag["longest"] = [
        {"name": r["name"], "duration": r["duration"], "peak_rss": r["peak_rss"]}
        for _, r in done.sort_values("realtime", ascending=False).head(10).iterrows()
    ]
    diag["workflow"] = {
        "total_tasks": int(len(tr)),
        "failed_tasks": int((tr["status"] == "FAILED").sum()),
        "retries": int((tr["attempt"] > 1).sum()),
    }
    return diag

In [ ]:
#| export
def build_report_data(
    outdir: str | Path,
    *,
    trace_path: str | Path | None = None,
    run_label: str = "",
) -> dict:
    """Assemble the full report data dict from a pipeline output directory.

    Aggregates-only by construction: per-sample arrays are consumed in memory and
    never emitted. Serialization must go through :func:`write_report_data` (or apply
    :func:`assert_no_phi`) so the no-identifiers guarantee is enforced.

    Args:
        outdir: The pipeline ``--outdir`` (contains ``labels/``, ``models/``,
            ``matrices/``, ``scoreboard_combined__all.parquet``, …).
        trace_path: Optional ``execution_trace.txt`` for the diagnostics tab.
        run_label: Free-text label shown in the report header.

    Raises:
        FileNotFoundError: if the scoreboard or labels are missing — the report is
            meaningless without them, so this fails loud rather than emitting an
            empty page.
    """
    outdir = Path(outdir)
    sb_path = outdir / "scoreboard_combined__all.parquet"
    labels_path = outdir / "labels" / "labels.parquet"
    for p, what in ((sb_path, "scoreboard"), (labels_path, "labels")):
        if not p.exists():
            raise FileNotFoundError(f"report_data: {what} not found at {p}")

    labels = pd.read_parquet(labels_path)
    lab_by_id = labels.set_index("SAMPLE_ID")
    sb = pd.read_parquet(sb_path).replace({np.nan: None})

    from kreview import __version__

    evaluators = [
        _build_evaluator(row, outdir, lab_by_id) for _, row in sb.iterrows()
    ]
    data = {
        "meta": {
            "title": "kreview evaluation report",
            "version": __version__,
            "run": run_label or outdir.name,
        },
        "cohort": _build_cohort(labels),
        "evaluators": evaluators,
        "multimodal": _build_multimodal(outdir / "models" / "multimodal"),
        "diagnostics": _build_diagnostics(trace_path),
    }
    log.info(
        "report_data_built",
        n_evaluators=len(evaluators),
        n_multimodal_models=len(data["multimodal"].get("models", {})),
        has_trace=bool(data["diagnostics"]["workflow"]),
    )
    return data


def write_report_data(data: dict, out_path: str | Path) -> Path:
    """Serialize report data with the PHI guarantee enforced."""
    blob = json.dumps(data, separators=(",", ":"), allow_nan=False)
    assert_no_phi(blob)
    out_path = Path(out_path)
    out_path.write_text(blob)
    log.info("report_data_written", path=str(out_path), kb=len(blob) // 1024)
    return out_path

In [ ]:
#| export
def render_page(data: dict, out_html: str | Path) -> Path:
    """Inject report data + inlined plotly.js into the page template and write it.

    Fully self-contained output (no CDN): plotly.js comes from the installed
    ``plotly`` python package, so the page works on air-gapped HPC nodes. The PHI
    guarantee is enforced on both the data blob and the final page.
    """
    from importlib.resources import files

    import plotly.offline as _po

    blob = json.dumps(data, separators=(",", ":"), allow_nan=False)
    assert_no_phi(blob)

    template = files("kreview.templates").joinpath("report_page.html").read_text()
    page = template.replace("__PLOTLY_JS__", _po.get_plotlyjs(), 1)
    page = page.replace("__REPORT_DATA__", blob, 1)
    assert_no_phi(page)

    out_html = Path(out_html)
    out_html.parent.mkdir(parents=True, exist_ok=True)
    out_html.write_text(page)
    log.info("report_rendered", path=str(out_html), mb=round(len(page) / 1e6, 1))
    return out_html


def render_report(
    outdir: str | Path,
    out_html: str | Path,
    *,
    trace_path: str | Path | None = None,
    run_label: str = "",
) -> Path:
    """Build the report data from ``outdir`` and render the single-page report."""
    data = build_report_data(outdir, trace_path=trace_path, run_label=run_label)
    return render_page(data, out_html)